In [ ]:
!mamba install Scikit-learn
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from scipy.optimize import curve_fit
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

def load_data(filename):
    try:
        data = np.load(filename)
        print(f"Данные успешно загружены из файла {filename}")
        print(f"Размерность данных: {data.shape}")
        return data
    except FileNotFoundError:
        print(f"Ошибка: Файл {filename} не найден!")
        return None

def detect_outliers(data):
    # Разделяем данные на X и Y (кроме последней строки)
    X_full = data[:-1, 0]  # все x кроме последнего
    y_full = data[:-1, 1]  # все y кроме последнего
    
    # Вычисляем Z-оценки для y
    z_scores = zscore(y_full)
    
    # Находим индексы выбросов (|z| >= 3)
    outlier_indices = np.where(np.abs(z_scores) >= 3)[0]
    
    # Создаем маску для фильтрации
    mask = np.ones(len(y_full), dtype=bool)
    mask[outlier_indices] = False
    
    # Данные без выбросов
    X_clean = X_full[mask]
    y_clean = y_full[mask]
    
    # Выбросы
    X_outliers = X_full[~mask]
    y_outliers = y_full[~mask]
    
    print(f"\nОбнаружено выбросов: {len(outlier_indices)}")
    if len(outlier_indices) > 0:
        print(f"Индексы выбросов: {outlier_indices}")
        print(f"Значения y-выбросов: {y_outliers}")
    
    return X_clean, y_clean, X_outliers, y_outliers

def manual_linear_regression(X, y):
    # Вычисляем средние значения
    x_mean = np.mean(X)
    y_mean = np.mean(y)
    x_squared_mean = np.mean(X ** 2)
    xy_mean = np.mean(X * y)
    
    # Расчет коэффициента k по формуле (4)
    k = (xy_mean - x_mean * y_mean) / (x_squared_mean - x_mean ** 2)
    
    # Расчет коэффициента b по формуле (5)
    b = y_mean - k * x_mean
    
    return k, b


def linear_func(x, k, b):
    return k * x + b

def scipy_linear_regression(X, y):
    popt, _ = curve_fit(linear_func, X, y)
    return popt[0], popt[1]  # k, b


def calculate_r2(y_true, y_pred):
    # Остаточная сумма квадратов
    ss_res = np.sum((y_true - y_pred) ** 2)
    # Общая сумма квадратов
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    # Коэффициент детерминации
    r2 = 1 - (ss_res / ss_tot)
    return r2


def plot_results(X_train, y_train, X_test, y_test, 
                X_outliers, y_outliers,
                k_manual, b_manual, 
                k_scipy, b_scipy,
                x_pred, y_pred_manual, y_pred_scipy,
                save_name='regression_plot.png'):
    plt.figure(figsize=(12, 8))
    
    # Обучающая выборка (синие точки)
    plt.scatter(X_train, y_train, c='blue', label='Обучающая выборка', alpha=0.6)
    
    # Тестовая выборка (зеленые точки, крупнее)
    plt.scatter(X_test, y_test, c='green', s=100, label='Тестовая выборка', alpha=0.7)
    
    # Выбросы (красные точки, крупнее)
    if len(X_outliers) > 0:
        plt.scatter(X_outliers, y_outliers, c='red', s=150, 
                   marker='X', label='Выбросы', alpha=0.8)
    
    # Линия регрессии (ручной расчет)
    X_line = np.linspace(min(X_train.min(), X_test.min()), 
                         max(X_train.max(), X_test.max(), x_pred), 100)
    y_line_manual = k_manual * X_line + b_manual
    plt.plot(X_line, y_line_manual, 'k-', linewidth=2, 
             label=f'Ручной расчет: y = {k_manual:.4f}x + {b_manual:.4f}')
    
    # Линия регрессии (scipy)
    y_line_scipy = k_scipy * X_line + b_scipy
    plt.plot(X_line, y_line_scipy, 'm--', linewidth=2, 
             label=f'Scipy curve_fit: y = {k_scipy:.4f}x + {b_scipy:.4f}')
    
    # Точка предсказания
    plt.scatter([x_pred], [y_pred_manual], c='purple', s=200, 
               marker='D', label=f'Предсказание (x={x_pred:.2f}, y={y_pred_manual:.4f})')
    
    plt.xlabel('X (независимая переменная)')
    plt.ylabel('Y (зависимая переменная)')
    plt.title('Парная линейная регрессия')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Сохраняем и показываем
    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()
    
    return plt

def run_experiment(data, X_clean, y_clean, X_outliers, y_outliers, x_pred, exp_num):

    print(f"ЭКСПЕРИМЕНТ №{exp_num}")
    
    # Разделение на обучающую и тестовую выборки (75% / 25%)
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, train_size=0.75, random_state=None
    )
    
    print(f"Обучающая выборка: {len(X_train)} точек")
    print(f"Тестовая выборка: {len(X_test)} точек")
    
    # Ручной расчет коэффициентов
    k_manual, b_manual = manual_linear_regression(X_train, y_train)
    print(f"\nРучной расчет: k = {k_manual:.6f}, b = {b_manual:.6f}")
    
    # Расчет через scipy
    k_scipy, b_scipy = scipy_linear_regression(X_train, y_train)
    print(f"Scipy curve_fit: k = {k_scipy:.6f}, b = {b_scipy:.6f}")
    
    # Разница в расчетах
    k_diff = abs(k_manual - k_scipy)
    b_diff = abs(b_manual - b_scipy)
    print(f"Разница: Δk = {k_diff:.2e}, Δb = {b_diff:.2e}")
    
    # Предсказание на тестовой выборке
    y_pred_test = k_manual * X_test + b_manual
    
    # Оценка качества
    r2_manual = calculate_r2(y_test, y_pred_test)
    r2_scikit = r2_score(y_test, y_pred_test)
    
    print(f"\nКоэффициент детерминации R²:")
    print(f"  Ручной расчет: {r2_manual:.6f}")
    print(f"  sklearn.metrics.r2_score: {r2_scikit:.6f}")
    
    # Предсказание для заданного x
    y_pred_value = k_manual * x_pred + b_manual
    print(f"\nПредсказание для x = {x_pred}: y = {y_pred_value:.6f}")
    
    # Визуализация
    plot_filename = f'regression_exp{exp_num}.png'
    plot_results(X_train, y_train, X_test, y_test,
                X_outliers, y_outliers,
                k_manual, b_manual,
                k_scipy, b_scipy,
                x_pred, y_pred_value, y_pred_value,
                plot_filename)
    
    return {
        'exp_num': exp_num,
        'k_manual': k_manual,
        'b_manual': b_manual,
        'k_scipy': k_scipy,
        'b_scipy': b_scipy,
        'r2': r2_manual,
        'y_pred': y_pred_value,
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test
    }

def main():
    # Загружаем данные
    filename = 'ml1var17.npy'
    data = load_data(filename)
    
    if data is None:
        return
    
    print(f"\nИсходные данные:")
    print(f"Первые 5 строк:\n{data[:5]}")
    print(f"Последняя строка (x для предсказания): {data[-1]}")
    
    # Получаем x для предсказания
    x_pred = data[-1, 0]
    print(f"x для предсказания = {x_pred}")
    
    # Обнаруживаем выбросы
    X_clean, y_clean, X_outliers, y_outliers = detect_outliers(data)
    
    # Проводим 3 эксперимента
    results = []
    for i in range(1, 4):
        result = run_experiment(data, X_clean, y_clean, 
                               X_outliers, y_outliers, x_pred, i)
        results.append(result)

    print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
    
    print(f"{'Эксп.':<6} {'k (ручной)':<15} {'b (ручной)':<15} {'k (scipy)':<15} {'b (scipy)':<15} {'R²':<10} {'Предсказание':<15}")

    for res in results:
        print(f"{res['exp_num']:<6} {res['k_manual']:<15.6f} {res['b_manual']:<15.6f} "
              f"{res['k_scipy']:<15.6f} {res['b_scipy']:<15.6f} {res['r2']:<10.6f} {res['y_pred']:<15.6f}")
    
    # Анализ различий

    print("АНАЛИЗ РАЗЛИЧИЙ В ЭКСПЕРИМЕНТАХ")
    
    k_values = [res['k_manual'] for res in results]
    b_values = [res['b_manual'] for res in results]
    r2_values = [res['r2'] for res in results]
    y_pred_values = [res['y_pred'] for res in results]
    
    print(f"Разброс k: от {min(k_values):.6f} до {max(k_values):.6f} (Δ = {max(k_values)-min(k_values):.6f})")
    print(f"Разброс b: от {min(b_values):.6f} до {max(b_values):.6f} (Δ = {max(b_values)-min(b_values):.6f})")
    print(f"Разброс R²: от {min(r2_values):.6f} до {max(r2_values):.6f}")
    print(f"Разброс предсказаний: от {min(y_pred_values):.6f} до {max(y_pred_values):.6f}")
    
    # Создаем финальный график со всеми линиями регрессии
    plt.figure(figsize=(14, 8))
    
    # Все точки (чистые данные)
    plt.scatter(X_clean, y_clean, c='lightblue', alpha=0.5, s=50, label='Все данные (без выбросов)')
    
    # Выбросы
    if len(X_outliers) > 0:
        plt.scatter(X_outliers, y_outliers, c='red', s=200, marker='X', label='Выбросы')
    
    # Линии регрессии для каждого эксперимента
    colors = ['blue', 'green', 'purple']
    for i, res in enumerate(results):
        X_line = np.linspace(X_clean.min(), X_clean.max(), 100)
        y_line = res['k_manual'] * X_line + res['b_manual']
        plt.plot(X_line, y_line, color=colors[i], linewidth=2, 
                label=f'Эксп.{res["exp_num"]}: R²={res["r2"]:.4f}')
    
    # Точка предсказания (среднее)
    avg_pred = np.mean(y_pred_values)
    plt.scatter([x_pred], [avg_pred], c='orange', s=300, marker='D', 
               edgecolors='black', linewidth=2, 
               label=f'Среднее предсказание: y={avg_pred:.4f}')
    
    plt.xlabel('X (независимая переменная)')
    plt.ylabel('Y (зависимая переменная)')
    plt.title('Сравнение линий регрессии в разных экспериментах')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('regression_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return results

if __name__ == "__main__":
    np.random.seed(None)  # Случайная инициализация для каждого запуска
    results = main()
    
    print("РАБОТА ЗАВЕРШЕНА")
    print("Результаты сохранены в файлах:")
    print("  - regression_exp1.png")
    print("  - regression_exp2.png")
    print("  - regression_exp3.png")
    print("  - regression_comparison.png")